# Station 8: Experimente mit Trainingsdaten und Hyperparametern

Informatik 13 · Künstliche Intelligenz · edu-mrh.de · CC BY-NC-SA 4.0

Daten: Palmer Penguins (Horst, Hill & Gorman 2020, Palmer Station LTER), CC0 1.0; Ziffern: E. Alpaydin, C. Kaynak (1998), Optical Recognition of Handwritten Digits, UCI Machine Learning Repository, doi:10.24432/C50P49, CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/), Testteil unverändert als CSV; übrige Datensätze erfunden.

Dieselben Programme wie auf der Website 8-experimente.html. Führe zuerst die Hilfszelle aus. Einrichtung von Python und Jupyter: https://edu-mrh.de/informatik/13/ki/python-einrichten.html

In [ ]:
# Hilfszelle: zuerst ausführen.
# Auf der Website zeichnet das Modul "ki" die Diagramme im Browser. Hier übernimmt das matplotlib.
# Datendateien (z. B. ring.csv) werden beim ersten Mal von edu-mrh.de geladen und neben dem Notebook gespeichert.
import os, urllib.request
import matplotlib.pyplot as plt

DATEN_URL = "https://edu-mrh.de/informatik/13/ki/data/"

def datei(name):
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATEN_URL + name, name)
    return name

class ki:
    @staticmethod
    def linie(x, reihen, titel="", xname="", yname=""):
        if not isinstance(reihen, dict):
            reihen = {yname or "Wert": list(reihen)}
        for name, werte in reihen.items():
            plt.plot(list(x), werte, marker="o" if len(werte) < 40 else None, label=name)
        if len(reihen) > 1:
            plt.legend()
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.grid(alpha=.3); plt.show()

    @staticmethod
    def streu(punkte, klassen=None, titel="", xname="x1", yname="x2"):
        klassen = klassen or [0] * len(punkte)
        plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.gca().set_aspect("equal"); plt.show()

    @staticmethod
    def region(f, punkte=(), klassen=None, schritte=40, titel="", xname="x1", yname="x2", bereich=(0, 1)):
        a, b = bereich
        gitter = [[f(a + (b - a) * (i + 0.5) / schritte, a + (b - a) * (j + 0.5) / schritte) for i in range(schritte)] for j in range(schritte)]
        plt.imshow(gitter, origin="lower", extent=(a, b, a, b), cmap="coolwarm", alpha=.35, vmin=0, vmax=1)
        if punkte:
            plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def balken(namen, werte, titel="", xname="", yname=""):
        plt.bar([str(n) for n in namen], werte); plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def tabelle(zeilen, kopf=None, titel=""):
        if titel:
            print(titel)
        if kopf:
            print(" | ".join(str(k) for k in kopf))
        for z in zeilen:
            print(" | ".join(str(round(v, 4)) if isinstance(v, float) else str(v) for v in z))

    @staticmethod
    def ziffern(bilder, namen=None, titel=""):
        fig, achsen = plt.subplots(1, len(bilder), figsize=(1.2 * len(bilder), 1.6))
        if len(bilder) == 1:
            achsen = [achsen]
        for k, (ax, bild) in enumerate(zip(achsen, bilder)):
            ax.imshow([bild[i * 8:(i + 1) * 8] for i in range(8)], cmap="gray_r", vmin=0, vmax=16)
            ax.axis("off")
            if namen:
                ax.set_title(str(namen[k]), fontsize=9)
        fig.suptitle(titel); plt.show()

for _name in ["pinguine.csv", "ziffern.csv", "ring.csv"]:
    try:
        datei(_name)
    except Exception as _fehler:
        print("Konnte", _name, "nicht laden:", _fehler)
print("Hilfszelle bereit.")


## Die Ziffern ansehen

**Idee:** Die Daten kennenlernen, bevor man ein Netz darauf loslässt.
                            
1. Jede Zeile der Datei: 64 Pixelwerte (0 = weiß bis 16 = schwarz) und die Ziffer.
1. Die ersten 12 Bilder werden gezeichnet, Bild 0 auch als Zahlen ausgegeben.
                            

**Achte auf:** Wie viele Bilder gibt es von jeder Ziffer? Erkennst du im Zahlenblock die Form der Ziffer?

In [ ]:
import csv
from collections import Counter

zeilen = list(csv.reader(open(datei("ziffern.csv"))))[1:]
bilder = [[int(v) for v in z[:64]] for z in zeilen]
ziffern = [int(z[64]) for z in zeilen]
print(len(bilder), "Bilder, je 8 x 8 = 64 Pixel (0 = weiß, 16 = schwarz)")
ki.ziffern(bilder[:12], ziffern[:12], titel="Die ersten 12 Bilder")
print("Bild 0 (eine", ziffern[0], ") als Zahlen:")
for r in range(8):
    print(" ".join("%2d" % v for v in bilder[0][r * 8:(r + 1) * 8]))
print("Bilder je Ziffer:", sorted(Counter(ziffern).items()))

## Ein Netz lernt Ziffern

**Idee:** Dasselbe Netz wie in Station 7, nur größer: 64 Eingaben, `VERDECKT` verdeckte Neuronen, 10 Ausgaben. Trainiert wird mit den ersten `TRAINING` Bildern, getestet mit dem Rest.
                            
1. Die Einstellungen stehen oben im Kasten: hier experimentierst du.
1. Nach jeder Epoche: richtig auf Training und Test.
1. Am Ende: Lernkurve und eine Tabelle, welche Ziffern verwechselt werden.
                            

**Achte auf:** Ändere jeweils einen Wert und vergleiche: `VERDECKT` = 2, 4, 12, 32 und `TRAINING` = 50, 200, 600, 1200. Wann klaffen Training und Test auseinander?

In [ ]:
import csv, math, random, time

random.seed(2)
zeilen = list(csv.reader(open(datei("ziffern.csv"))))[1:]
daten = [([int(v) / 16 for v in z[:64]], int(z[64])) for z in zeilen]
random.shuffle(daten)

# ---------- Einstellungen für deine Experimente ----------
VERDECKT = 12      # Neuronen in der verdeckten Schicht
TRAINING = 600     # Zahl der Trainingsbilder (der Rest bis 1797 sind Testdaten)
EPOCHEN = 8
ETA = 0.5          # Lernrate
# ---------------------------------------------------------
train, test = daten[:TRAINING], daten[TRAINING:]

def sigmoid(z):
    if z < -60:
        return 0.0
    return 1 / (1 + math.exp(-z))

class Netz:
    def __init__(self, groessen):
        self.W = []
        for l in range(1, len(groessen)):
            self.W.append([[random.uniform(-0.5, 0.5) for i in range(groessen[l - 1] + 1)] for j in range(groessen[l])])

    def vorwaerts(self, x):
        a = [x]
        for schicht in self.W:
            vorher = a[-1]
            a.append([sigmoid(w[0] + sum(wi * ai for wi, ai in zip(w[1:], vorher))) for w in schicht])
        return a

    def rueckwaerts(self, x, ziel, eta):
        a = self.vorwaerts(x)
        delta = [[2 * (y - t) * y * (1 - y) for y, t in zip(a[-1], ziel)]]
        for l in range(len(self.W) - 1, 0, -1):
            delta.insert(0, [sum(self.W[l][k][j + 1] * delta[0][k] for k in range(len(self.W[l]))) * a[l][j] * (1 - a[l][j]) for j in range(len(self.W[l - 1]))])
        for l, schicht in enumerate(self.W):
            for j, w in enumerate(schicht):
                d = eta * delta[l][j]
                w[0] -= d
                for i, ai in enumerate(a[l]):
                    w[i + 1] -= d * ai

    def erkenne(self, x):
        y = self.vorwaerts(x)[-1]
        return y.index(max(y))

def genauigkeit(netz, menge):
    return sum(1 for x, z in menge if netz.erkenne(x) == z) / len(menge)

netz = Netz([64, VERDECKT, 10])
start = time.time()
kurve_train, kurve_test = [], []
for e in range(1, EPOCHEN + 1):
    for x, z in train:
        netz.rueckwaerts(x, [1 if k == z else 0 for k in range(10)], ETA)
    kurve_train.append(100 * genauigkeit(netz, train))
    kurve_test.append(100 * genauigkeit(netz, test))
    print("Epoche", e, "  richtig: Training", round(kurve_train[-1]), "%   Test", round(kurve_test[-1]), "%")
print("Zeit:", round(time.time() - start, 1), "s   Gewichte im Netz:", sum(len(w) for s in netz.W for w in s))
ki.linie(range(1, EPOCHEN + 1), {"Training": kurve_train, "Test": kurve_test}, titel="Richtig erkannt (%)", xname="Epoche", yname="%")

verwechselt = {}
for x, z in test:
    e = netz.erkenne(x)
    if e != z:
        verwechselt[(z, e)] = verwechselt.get((z, e), 0) + 1
oft = sorted(verwechselt.items(), key=lambda p: -p[1])[:6]
ki.tabelle([[z, e, n] for (z, e), n in oft], ["Ziffer", "erkannt als", "wie oft"], titel="Häufigste Verwechslungen (Testdaten)")

## Unausgewogene Trainingsdaten

**Idee:** Dasselbe Netz, aber von der Ziffer 8 bleiben im Training nur wenige Bilder übrig. Danach wird für jede Ziffer einzeln gemessen, wie oft sie in den Testdaten richtig erkannt wird.
                                

**Achte auf:** Vergleiche „richtig insgesamt“ mit dem Balken der 8. Ändere `ACHTEN` auf 15 und 56 (alle).

In [ ]:
import csv, math, random

random.seed(2)
zeilen = list(csv.reader(open(datei("ziffern.csv"))))[1:]
daten = [([int(v) / 16 for v in z[:64]], int(z[64])) for z in zeilen]
random.shuffle(daten)
train, test = daten[:600], daten[600:]

ACHTEN = 5         # so viele Achten bleiben im Training (von 56)
achten = [d for d in train if d[1] == 8]
train = [d for d in train if d[1] != 8] + achten[:ACHTEN]
random.seed(5)
random.shuffle(train)

def sigmoid(z):
    if z < -60:
        return 0.0
    return 1 / (1 + math.exp(-z))

random.seed(2)
W = [[[random.uniform(-0.5, 0.5) for i in range(65)] for j in range(12)],
     [[random.uniform(-0.5, 0.5) for i in range(13)] for j in range(10)]]

def vorwaerts(x):
    h = [sigmoid(w[0] + sum(wi * xi for wi, xi in zip(w[1:], x))) for w in W[0]]
    y = [sigmoid(w[0] + sum(wi * hi for wi, hi in zip(w[1:], h))) for w in W[1]]
    return h, y

def lerne(x, ziffer, eta=0.5):
    h, y = vorwaerts(x)
    d_aus = [2 * (y[k] - (1 if k == ziffer else 0)) * y[k] * (1 - y[k]) for k in range(10)]
    d_ver = [sum(W[1][k][j + 1] * d_aus[k] for k in range(10)) * h[j] * (1 - h[j]) for j in range(12)]
    for k, w in enumerate(W[1]):
        w[0] -= eta * d_aus[k]
        for j in range(12):
            w[j + 1] -= eta * d_aus[k] * h[j]
    for j, w in enumerate(W[0]):
        w[0] -= eta * d_ver[j]
        for i in range(64):
            w[i + 1] -= eta * d_ver[j] * x[i]

def erkenne(x):
    y = vorwaerts(x)[1]
    return y.index(max(y))

for epoche in range(8):
    for x, z in train:
        lerne(x, z)

je_ziffer = []
for ziffer in range(10):
    menge = [x for x, z in test if z == ziffer]
    je_ziffer.append(round(100 * sum(1 for x in menge if erkenne(x) == ziffer) / len(menge)))
gesamt = sum(1 for x, z in test if erkenne(x) == z) / len(test)
print("Achten im Training:", ACHTEN, "  richtig insgesamt:", round(100 * gesamt), "%")
print("richtig je Ziffer (%):", je_ziffer)
ki.balken(range(10), je_ziffer, titel="Richtig erkannt je Ziffer (Testdaten, %)", xname="Ziffer", yname="%")